In [ ]:
import requests
import fitz  # pymupdf
from bs4 import BeautifulSoup
from openai import OpenAI
import os
import datetime
import re
import json
import markdown
from weasyprint import HTML
from dotenv import load_dotenv

# Load env variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)
LLM_MODEL = "gpt-4o-mini"
OUTPUT_DIR = "deepdives"

# ------------- HELPER FUNCTIONS ----------------

def _create_anchor_slug(title):
    """Creates a URL-friendly slug from a title for anchor links (Hugo style)."""
    s = title.lower()
    s = re.sub(r'[^\w\s-]', '', s)  # Remove non-alphanumeric characters
    s = re.sub(r'[\s_-]+', '-', s).strip('-')  # Replace spaces with hyphens
    return s

def fetch_paper_text(url):
    """Downloads paper and extracts text using PyMuPDF."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    print(f"📥 Fetching: {url}")
    pdf_url = url
    
    try:
        # 1. Handle ArXiv
        if "arxiv.org/abs" in url:
            pdf_url = url.replace("/abs/", "/pdf/")
            if not pdf_url.endswith(".pdf"): pdf_url += ".pdf"
            
        # 2. Handle Landing Pages
        elif not url.lower().endswith(".pdf"):
            try:
                response = requests.get(url, headers=headers, timeout=10)
                soup = BeautifulSoup(response.text, "html.parser")
                found_link = soup.find('a', href=True, string=lambda t: t and ("pdf" in t.lower() or "download" in t.lower()))
                if not found_link:
                    found_link = soup.find('a', href=lambda h: h and ".pdf" in h.lower())
                
                if found_link:
                    href = found_link['href']
                    if href.startswith("http"):
                        pdf_url = href
                    elif href.startswith("/"):
                        from urllib.parse import urlparse
                        parsed = urlparse(url)
                        pdf_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                    else:
                        pdf_url = url.rstrip("/") + "/" + href
                    print(f"   -> Detected PDF Link: {pdf_url}")
            except Exception as e:
                print(f"   ⚠️ Could not scrape landing page: {e}. Trying original URL.")

        # 3. Download
        response = requests.get(pdf_url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # 4. Extract
        with fitz.open(stream=response.content, filetype="pdf") as doc:
            text = ""
            meta_title = doc.metadata.get('title', '')
            if not meta_title or meta_title.strip() == "":
                meta_title = "Unknown Title"
            
            # Read first 30 pages
            for page in doc[:30]:
                text += page.get_text()
                
        return text, meta_title

    except Exception as e:
        return f"[ERROR] {e}", "Error"

def generate_deep_dive_json(text, meta_title, original_url):
    """
    Sends text to LLM and requests a JSON response containing the Title, Authors, and Analysis.
    This ensures we have clean metadata for the Table of Contents.
    """
    if text.startswith("[ERROR]"):
        return {
            "title": "Error Processing Paper",
            "authors": "Unknown",
            "analysis": f"Could not extract text: {text}"
        }

    prompt = f"""
You are a highly specialized expert curator for **“ets4 Monthly (Economic Time Series Forecasting Monthly),”** a newsletter focused exclusively on **practical and impactful forecasting of economic time series.** 
You are also a seasoned **econometrician, forecaster, and data scientist** with deep experience evaluating new modeling techniques, understanding their assumptions and limits, and interpreting empirical evidence.

Your task is to identify the paper's metadata and produce a **technical Deep Dive** into the research paper provided below.

**OUTPUT FORMAT INSTRUCTIONS:**
You must strictly return a **valid JSON object** containing the following keys:
1. `"title"`: The exact title of the paper.
2. `"authors"`: A string listing the authors.
3. `"analysis"`: A Markdown-formatted string containing the Deep Dive report.

**INSTRUCTIONS FOR THE "analysis" CONTENT:**

Before producing the analysis string, apply the following critical lens:

• **Introduce the foundational concepts** necessary to understand the paper’s contribution.
  – Provide a *brief informal explanation* (intuition first).  
  – Provide a *concise formal explanation* (a few key equations only).  
  – These foundations may be broader than the specific innovation, and should orient a researcher not specialized in this sub-field.

• When analyzing the core idea of the paper, **focus on the intuition, design choices, assumptions, and where the method works or fails**, rather than long derivations.  
  – Highlight examples, counterexamples, and scenarios where the method is brittle or where assumptions are unrealistic.

• Provide **two–three sentences on the literature gap** the paper fills and why it matters.

• Read empirical sections *critically*:  
  – Extract findings from tables and figures.  
  – Pay attention to situations where the method *underperforms*, including in Monte Carlo experiments (and whether the MC design is sound).  
  – Emphasize practical implications and caveats for real-world forecasters.

**STRUCTURE FOR THE "analysis" STRING (Strictly follow this):**

1. **Foundational Concepts (Informal + Formal):**  
   The minimum background a non-specialist economist/forecaster needs to follow the paper. Include 1–3 key formulas max.

2. **The Core Innovation:**  
   The specific modeling contribution. Explain the intuition, what it tries to fix, the assumptions under which it works, and importantly **when and why it may fail**.

3. **Methodology:**  
   Model architecture, estimation strategy, feature engineering, experimental setup. Keep it clear and emphasize design decisions over math. Include 1–3 key formulas max.

4. **Position in the Literature:**  
   Briefly state (2–3 sentences) what gap the paper fills.

5. **Empirical Evidence:**  
   Dataset(s), forecasting setup, evaluation metrics, performance vs. baselines.  
   Highlight **both strengths and weak points**, especially where tables/figures reveal weaknesses or instability.

6. **Critical Takeaway (for Practitioners):**  
   One sentence on why this paper matters—or why it doesn’t—for real-world economic time-series forecasting.

---
**TEXT START**
{text[:80000]} 
**TEXT END**
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return data
    except Exception as e:
        return {
            "title": meta_title,
            "authors": "Unknown",
            "analysis": f"LLM Generation Error: {e}"
        }

# ------------- OUTPUT GENERATION ----------------

def save_outputs(analyzed_papers, output_formats=['md', 'html', 'pdf']):
    """
    Generates the formatted report in requested formats.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today = datetime.date.today()
    
    # Date Formats
    iso_date_str = today.strftime('%Y-%m-%d')
    main_header_date_str = today.strftime('%B %d, %Y')
    front_matter_date_str = today.strftime('%B %Y')
    
    base_filename = f"ets4_deepdive_monthly_{iso_date_str}"
    
    # --- 1. BUILD MARKDOWN CONTENT (Hugo Style) ---
    
    front_matter = f"""---
title: "ets4 Deep Dive: {front_matter_date_str}"
date: {iso_date_str}
draft: true
toc: false
---
"""
    lines = [
        front_matter,
        f"# ets4 Deep Dive: {main_header_date_str}\n",
        "\n&nbsp;\n"
    ]
    
    # TOC Section
    if analyzed_papers:
        lines.append("## In this Issue\n")
        for p in analyzed_papers:
            anchor = _create_anchor_slug(p['title'])
            lines.append(f"* [{p['title']}](#{anchor})")
        lines.append("\n---\n")
    
    # Body Section
    for p in analyzed_papers:
        anchor = _create_anchor_slug(p['title'])
        lines.append(f"## {p['title']} {{#{anchor}}}") # Hugo syntax for explicit id, or standard markdown
        # Fallback standard markdown anchor if Hugo processing isn't guaranteed in raw view:
        # But for standard markdown rendering, the header text is the anchor.
        
        lines.append(f"[Link to Source ↗]({p['link']})\n")
        lines.append(f"**Authors:** {p['authors']}\n")
        lines.append(p['analysis'])
        lines.append("\n&nbsp;\n")
        lines.append("\n---\n")

    full_markdown = "\n".join(lines)

    # Save Markdown
    if 'md' in output_formats:
        md_path = os.path.join(OUTPUT_DIR, base_filename + ".md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(full_markdown)
        print(f"✅ Markdown saved: {md_path}")

    # --- 2. BUILD HTML CONTENT ---
    if 'html' in output_formats or 'pdf' in output_formats:
        # Convert MD to HTML with extensions for tables and code blocks
        html_body = markdown.markdown(full_markdown, extensions=['tables', 'fenced_code'])
        
        # Add simple CSS for clean reading
        html_content = f"""
        <html>
        <head>
            <meta charset="utf-8">
            <style>
                body {{ font-family: 'Helvetica', 'Arial', sans-serif; line-height: 1.6; max-width: 800px; margin: auto; padding: 2em; color: #333; }}
                h1 {{ color: #2c3e50; border-bottom: 2px solid #eee; }}
                h2 {{ color: #2980b9; margin-top: 2em; }}
                h3 {{ color: #34495e; }}
                pre {{ background: #f4f4f4; padding: 1em; overflow-x: auto; border-radius: 4px; }}
                code {{ background: #f4f4f4; padding: 2px 5px; }}
                a {{ color: #3498db; text-decoration: none; }}
                blockquote {{ border-left: 4px solid #ddd; padding-left: 1em; color: #777; }}
            </style>
        </head>
        <body>
            {html_body}
        </body>
        </html>
        """
        
        if 'html' in output_formats:
            html_path = os.path.join(OUTPUT_DIR, base_filename + ".html")
            with open(html_path, "w", encoding="utf-8") as f:
                f.write(html_content)
            print(f"✅ HTML saved: {html_path}")

        # --- 3. BUILD PDF CONTENT ---
        if 'pdf' in output_formats:            
            pdf_path = os.path.join(OUTPUT_DIR, base_filename + ".pdf")
            HTML(string=html_content).write_pdf(pdf_path)
            print(f"✅ PDF saved: {pdf_path}")

# ------------- MAIN EXECUTION ----------------

def deepdive(links_vector, formats=['md', 'pdf']):
    """
    Main orchestrator.
    links_vector: List of URLs
    formats: List of strings ['md', 'html', 'pdf']
    """
    print(f"🚀 Starting Deep Dive for {len(links_vector)} papers...")
    
    analyzed_data = []
    
    for i, link in enumerate(links_vector, 1):
        print(f"\n--- Processing Paper {i}/{len(links_vector)} ---")
        
        # 1. Fetch Text
        text, meta_title = fetch_paper_text(link)
        
        if len(text) < 500:
            print(f"   ⚠️ Text too short for {link}. Skipping analysis.")
            continue
            
        # 2. Analyze with LLM (returns JSON)
        print(f"   🧠 Analyzing content...")
        result = generate_deep_dive_json(text, meta_title, link)
        
        # 3. Store result
        analyzed_data.append({
            "title": result.get("title", meta_title),
            "authors": result.get("authors", "Unknown"),
            "link": link,
            "analysis": result.get("analysis", "Analysis failed.")
        })

    # 4. Generate Files
    if analyzed_data:
        print("\n💾 Generating Reports...")
        save_outputs(analyzed_data, output_formats=formats)
    else:
        print("No papers were successfully analyzed.")

if __name__ == "__main__":
    # Example Links
    my_links = [
        "https://arxiv.org/pdf/2511.07014",
        "https://www.arxiv.org/abs/2511.07678",
    ]
    
    # You can specify formats: ['md', 'html', 'pdf']
    deepdive(my_links, formats=['md', 'html', 'pdf'])

🚀 Starting Deep Dive for 2 papers...

--- Processing Paper 1/2 ---
📥 Fetching: https://arxiv.org/pdf/2511.07014
   🧠 Analyzing content...

--- Processing Paper 2/2 ---
📥 Fetching: https://www.arxiv.org/abs/2511.07678
   🧠 Analyzing content...

💾 Generating Reports...
✅ Markdown saved: deepdives/ets4_deepdive_monthly_2025-11-25.md
✅ HTML saved: deepdives/ets4_deepdive_monthly_2025-11-25.html
✅ PDF saved: deepdives/ets4_deepdive_monthly_2025-11-25.pdf
